In [1]:
import os
import pandas as pd
from copy import deepcopy
from omegaconf import DictConfig
from pathlib import Path

import tiktoken
from chromadb import ClientAPI, Collection, PersistentClient, QueryResult
from datasets import Dataset, DatasetDict, load_dataset
from hydra import compose, initialize
from pyrootutils import setup_root
from sklearn.model_selection import train_test_split

/Users/pacebailey/Documents/Education/Universität Potsdam/IM/relation-extraction/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
search_from: str = os.getcwd()
root: Path = setup_root(search_from)
with initialize(config_path="config", version_base=None):
    config: DictConfig = compose("config")

# Dataset Functions

In [3]:
def collate_documents(dataset: DatasetDict) -> Dataset:
    documents: list[dict] = [doc for split in dataset.values() for doc in split.to_list()]
    
    return Dataset.from_list(documents)

def remove_duplicates(dataset: Dataset) -> Dataset:
    dataframe: pd.DataFrame = pd.DataFrame(dataset)
    dataframe = dataframe.drop_duplicates(subset=["token"])

    return Dataset.from_pandas(dataframe)

def remove_extra_columns(dataset: Dataset, columns: list[str]) -> Dataset:
    all_columns: set[str] = set(dataset.column_names)
    extra_columns: set[str] = all_columns - set(columns)

    return dataset.remove_columns(list(extra_columns))

def group_spans(doc: dict) -> list[dict]:
    head_span: dict = {"tag": "HEAD", "relation": doc["relation"], "ner": doc["subj_type"], "start": doc["subj_start"], "end": doc["subj_end"]}
    tail_span: dict = {"tag": "TAIL", "relation": doc["relation"], "ner": doc["obj_type"], "start": doc["obj_start"], "end": doc["obj_end"]}

    return sorted([tail_span, head_span], key=lambda s: -s["end"])

def insert_entity_tags(tokens: list[str], spans: list[dict], format: str) -> list[str]:
    beginning, ending = tuple(format.split("{entity}"))
    if not beginning and not ending:
        return tokens

    spans_copy: list[dict] = deepcopy(spans)
    labeled_tokens: list[str] = tokens.copy()
    for i, span in enumerate(spans_copy):
        labeled_tokens.insert(span["end"] + 1, ending.format(**span))
        labeled_tokens.insert(span["start"], beginning.format(**span))
        span["start"] += 1
        span["end"] += 1
        for j in range(i):
            spans_copy[j]["start"] += 2
            spans_copy[j]["end"] += 2

    return labeled_tokens

def label_document(document: dict, input_format: str, output_format: str) -> dict[str, str]:
    spans: list[dict] = group_spans(document)
    input_tokens: list[str] = insert_entity_tags(document["token"], spans, input_format)
    output_tokens: list[str] = insert_entity_tags(document["token"], spans, output_format)
    
    return {"text": " ".join(document["token"]), "input": " ".join(input_tokens), "output": " ".join(output_tokens)}

def preprocess(dataset: DatasetDict, input_format: str, output_format: str, columns: list[str], random_state: int = 42) -> DatasetDict:
    collated: Dataset = collate_documents(dataset)
    unique: Dataset = remove_duplicates(collated)
    trimmed: Dataset = remove_extra_columns(unique, columns)
    labeled: Dataset = trimmed.map(label_document, fn_kwargs={"input_format": input_format, "output_format": output_format}, desc="Labeling documents")
    train, test = train_test_split(labeled.to_list(), random_state=random_state, stratify=list(labeled["relation"]))
    
    return DatasetDict({"train": Dataset.from_list(train), "test": Dataset.from_list(test)})

In [4]:
columns: list[str] = ["id", "relation", "token", "subj_start", "subj_end", "obj_start", "obj_end", "subj_type", "obj_type", "stanford_ner"]
input_format: str = "{entity}"
output_format: str = "<{tag} ner='{ner}' relation='{relation}'>{entity}</{tag}>"
dataset: DatasetDict = load_dataset(config.dataset.path, data_dir=config.dataset.data_dir)
preprocessed_dataset: DatasetDict = preprocess(dataset, input_format, output_format, columns)
print(preprocessed_dataset)

Labeling documents: 100%|██████████| 53791/53791 [00:03<00:00, 15784.82 examples/s]


DatasetDict({
    train: Dataset({
        features: ['id', 'relation', 'token', 'subj_start', 'subj_end', 'obj_start', 'obj_end', 'subj_type', 'obj_type', 'stanford_ner', 'text', 'input', 'output'],
        num_rows: 40343
    })
    test: Dataset({
        features: ['id', 'relation', 'token', 'subj_start', 'subj_end', 'obj_start', 'obj_end', 'subj_type', 'obj_type', 'stanford_ner', 'text', 'input', 'output'],
        num_rows: 13448
    })
})


# Vectorstore Functions

In [5]:
def get_metadata(doc: dict) -> dict:
    return {"relation": doc["relation"], "subj_type": doc["subj_type"], "obj_type": doc["obj_type"], "input": doc["input"], "output": doc["output"]}

def add_documents(collection: Collection, dataset: Dataset, batch_size: int = 5461) -> None:
    ids: list[int] = list(dataset["id"])
    documents: list[str] = list(dataset["text"])
    metadata: list[dict] = [get_metadata(document) for document in dataset]
    for idx in range(0, len(ids), batch_size):
        batch_end: int = idx + batch_size
        ids: list[str] = [str(id) for id in ids[idx:batch_end]]
        documents: list[str] = documents[idx:batch_end]
        metadatas: list[dict] = metadata[idx:batch_end]
        collection.upsert(ids, documents=documents, metadatas=metadatas)

def get_collection(dataset: Dataset, client: ClientAPI) -> Collection:
    collection: Collection = client.get_or_create_collection(name="ner")
    if not collection.count():
        add_documents(collection, dataset)

    return collection

In [6]:
client: ClientAPI = PersistentClient(root / "chroma")
collection: Collection = get_collection(preprocessed_dataset["train"], client)

# Prompt Functions

In [7]:
def configure_prompt(config: DictConfig, examples: QueryResult, document: dict) -> str:
    task: str = config.prompt.task.format(relation_type=document["relation"], head_type=document["subj_type"], tail_type=document["obj_type"], description=config.dataset.label_types[document["relation"]]["description"])
    prompt: str = f"{config.prompt.system} {task}"
    for metadatas in examples["metadatas"]:
        for metadata in metadatas:
            prompt += config.prompt.example.format(input=metadata["input"], output=metadata["output"])

    return prompt

def prompt_model(document: dict, collection: Collection, config: DictConfig) -> dict:
    filter: dict = {"$and": [{"relation": document["relation"]}, {"subj_type": document["subj_type"]}, {"obj_type": document["obj_type"]}]}
    examples: QueryResult = collection.query(query_texts=document["text"], n_results=config.rag.n_queries, where=filter)
    
    return configure_prompt(config, examples, document)

def count_tokens(prompt: str, config: DictConfig) -> int:
    encoding = tiktoken.encoding_for_model(config.model.model)
    tokens: list[str] = encoding.encode(prompt)

    return len(tokens)

In [8]:
token_count: int = 0
for document in preprocessed_dataset["test"]:
    prompt: str = prompt_model(document, collection, config)
    token_count += count_tokens(prompt, config)

print(token_count)

17033360
